In [1]:
%store -r

In [2]:
import commute_dm.core
import commute_dm.ig
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
def make_backend():
    return momapy_kb.lpg.backends.neo4j.Neo4jBackend(
        hostname=credentials.NEO4J_URI,
        username=credentials.NEO4J_USERNAME,
        password=credentials.NEO4J_PASSWORD,
        notifications_min_severity="off",
    )

In [4]:
MAX_LEVELS = [2, 3, 4, 5, 6]
MIN_N_NODES = 5
UPSTREAM_COLLECTION_NAME = commute_dm.core.UPSTREAM_COLLECTION_NAME
DOWNSTREAM_COLLECTION_NAME = commute_dm.core.DOWNSTREAM_COLLECTION_NAME

We compute the interface (between the two activity-flow collections and the AD BEL KG), and build the adjacency index over the stored activity-flow structure. The index is built **once per session** and threaded through the analysis.

In [5]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    interface = commute_dm.core.get_interface(session)
    index = commute_dm.ig.load_af_index(
        session, [UPSTREAM_COLLECTION_NAME, DOWNSTREAM_COLLECTION_NAME]
    )
len(interface), len(index.species), len(index.modulation_endpoints), len(index.gates)

(101, 4884, 5212, 35)

In [6]:
commute_dm.utils.remake_dir(INTERFACE_ANALYSIS_GRAPHS_DIR)

We select and render the sub-maps upstream of the COVID seeds and downstream of the PD seeds. Every element in the output is a **stored** activity-flow element (species, signed modulation, boolean logic gate, glyph, arc), except the synthetic central node standing for the interface protein itself.

In [7]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    stats_df = commute_dm.core.make_and_write_cd_maps_from_interface(
        session=session,
        interface=interface,
        index=index,
        output_dir_path=INTERFACE_ANALYSIS_GRAPHS_DIR,
        upstream_collection_name=UPSTREAM_COLLECTION_NAME,
        downstream_collection_name=DOWNSTREAM_COLLECTION_NAME,
        max_levels=MAX_LEVELS,
        upstream_nodes_color="lightblue",
        downstream_nodes_color="lightgreen",
        common_nodes_color="goldenrod",
        interface_nodes_color="red",
        min_n_nodes=MIN_N_NODES,
        include_compartment_layouts=True,
    )
stats_df

,identifier,display_name,max_level,n_species,n_modulations,n_gates,n_compartments,n_compartment_layouts,n_templates,n_renumbered,n_elements_without_glyph,n_modulations_dropped_no_glyph,n_extra_influences_dropped,n_subunit_entries,n_interned_away,n_colored
0,P45983,MAPK8,2,45,56,0,9,8,47,405,1,0,2,30,0,45
1,P45983,MAPK8,3,94,138,0,13,12,73,858,1,0,2,69,0,94
2,P45983,MAPK8,4,127,184,0,17,16,84,1116,1,0,2,86,0,127
3,P45983,MAPK8,5,182,285,1,21,20,134,1721,1,0,2,149,0,182
4,P45983,MAPK8,6,252,432,1,28,27,202,2504,1,0,2,215,0,252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,P05231,IL6,2,24,31,0,7,6,7,162,1,0,1,0,0,24
98,P05231,IL6,3,52,72,1,13,12,30,468,1,0,1,36,0,52
99,P05231,IL6,4,97,161,1,16,15,55,855,1,0,1,51,0,97
100,P05231,IL6,5,158,288,2,20,19,95,1468,1,0,1,97,0,158
